In [13]:
# conda env 'at3d_pure_k9'
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent

import at3d
from collections import OrderedDict
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

In [14]:
save_LUT = False

EXPERIMENT = PROJECT_ROOT / 'experiments' / 'CAMP2Ex_RF17' / 'V08' / 'default'
ITERATIONS = EXPERIMENT / 'iterations'
LUT_OUT = EXPERIMENT / 'tau_1D'
LUT_OUT.mkdir(parents=True, exist_ok=True)
LUT_outfile = LUT_OUT / 'LUT_1Dtau.nc'

iter_files = sorted(ITERATIONS.glob('iter*.nc'))
iter_file = iter_files[-1]
sensors, solvers, rte_grid = at3d.util.load_forward_model(iter_file)

/data/keeling/a/johntl3/miniconda3/envs/at3d_pure_k9/lib/python3.10/site-packages/at3d/solver.py:2337: UserWarning: Number of SHDOM grid cells with optical depth greater than 2: '71'. Max cell optical depth: '3.4591331481933594'
  warnings.warn("Number of SHDOM grid cells with optical depth greater than 2: '{}'. "


In [15]:
# fm_file = PROJECT_ROOT / 'experiments' / 'CAMP2Ex_RF17' / 'V07' / 'default' / 'iterations' / 'iter012.nc'

# save_LUT = True
# LUT_outfile = PROJECT_ROOT / 'NK1990_1D_Retrieval' / 'LUT_V07.nc'

# sensors, solvers, rte_grid = at3d.util.load_forward_model(fm_file)

wavelength = sensors.get_unique_solvers()
assert len(wavelength == 1)
wavelength = wavelength[0]

## Create rte_grid with same vertical grid, but only a 3 x 3 in the horizontal
rte_grid = at3d.grid.make_grid(rte_grid.delx.data, 3, rte_grid.dely.data, 3, rte_grid.z.data)

## Define cloud base and cloud top
cb = 0.4
ct = 0.7
i_low = np.argwhere(rte_grid.z.data >= cb)[0][0]
i_up = np.argwhere(rte_grid.z.data <= ct)[-1][0]
thickness = rte_grid.z.data[i_up] - rte_grid.z.data[i_low]
## Define optical depths in LUT
tausA = np.linspace(0, 4.75, 20)
tausB = np.logspace(np.log10(5), 2, 40)
taus = np.append(tausA, tausB)

In [16]:
## Make a sensor to observe radiance
LUT_sensors = at3d.containers.SensorsDict()
LUT_sensor = at3d.sensor.orthographic_projection(
    wavelength=wavelength,
    bounding_box=rte_grid,
    x_resolution=rte_grid.delx.data,
    y_resolution=rte_grid.dely.data,
    azimuth=0,
    zenith=0,
    altitude='TOA',
    stokes='I',
    sub_pixel_ray_args={'method': None}
)
LUT_sensors.add_sensor('LUT_sensor', LUT_sensor)

In [17]:
## Set up LUT solver with the features of the retreival solver

# Numerical parameters
numerical_parameters = solvers[wavelength].numerical_params
numerical_parameters['ip_flag'] = 3 # Set to independent pixel approximation
numerical_parameters['split_accuracy'] = 1e-3
# Surface
surface = solvers[wavelength].surface

# Source
source = solvers[wavelength].source

# Medium
medium = OrderedDict()
medium['background'] = solvers[wavelength].medium['background'].isel(x=slice(0,3), y=slice(0,3))
medium['cloud'] = solvers[wavelength].medium['cloud'].isel(x=slice(0,3), y=slice(0,3))

In [18]:
Is = np.zeros(taus.shape)
num_unique = np.zeros(taus.shape)
diff = np.zeros(taus.shape)
LUT_solvers = at3d.containers.SolversDict()
for i, tau in enumerate(taus):
    ext = np.zeros((rte_grid.dims['x'], rte_grid.dims['y'], rte_grid.dims['z']))
    ext[:, :, i_low:i_up] = tau/thickness
    medium['cloud']['extinction'] = (['x','y','z'], ext)

    LUT_solver = at3d.solver.RTE(
        numerical_params = numerical_parameters,
        medium = medium,
        source = source,
        surface = surface,
        num_stokes = 1
    )
    LUT_solvers.add_solver(wavelength, LUT_solver)
    LUT_sensors.get_measurements(LUT_solvers)

    cam = LUT_sensors['LUT_sensor']['sensor_list'][0]
    num_unique[i] = len(np.unique(cam.I.data))
    diff[i] = np.max(cam.I.data) - np.min(cam.I.data)
    # assert num_unique[i] <= 3, "Tau = {}\n{}".format(tau, cam.I)
    Is[i] = np.mean(cam.I.data)

  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -0.985  0.411E-01      135   256.00  1.000   [Radiance 0.670 micron]
     2  -1.442  0.420E-01      135   256.00  1.000   [Radiance 0.670 micron]
     3  -1.970  0.114E-01      144   256.00  1.000   [Radiance 0.670 micron]
     4  -1.963  0.111E-01      144   256.00  1.000   [Radiance 0.670 micron]
     5  -2.687  0.296E-02      171   256.00  1.000   [Radiance 0.670 micron]
     6  -2.553  0.745E-03      216   256.00  1.000   [Radiance 0.670 micron]
     7  -2.799  0.728E-03      216   256.00  1.000   [Radiance 0.670 micron]
     8  -3.098  0.731E-03      216   256.00  1.000   [Radiance 0.670 micron]
     9  -4.179  0.731E-03      216   256.00  1.000   [Radiance 0.670 micron]
 ! Iterations:      9     Final Criterion:  0.000066   [Radiance 0.670 micron]
  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -0.982  0.413E-01      135   256.00  1.000   [Radiance 0.670 micron]
   

/data/keeling/a/johntl3/miniconda3/envs/at3d_pure_k9/lib/python3.10/site-packages/at3d/solver.py:2337: UserWarning: Number of SHDOM grid cells with optical depth greater than 2: '8'. Max cell optical depth: '2.0755248069763184'
  warnings.warn("Number of SHDOM grid cells with optical depth greater than 2: '{}'. "


  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -0.880  0.169E+00      135   256.00  1.000   [Radiance 0.670 micron]
     2  -1.432  0.178E+00      135   256.00  1.000   [Radiance 0.670 micron]
     3  -1.617  0.184E-01      216   256.00  1.000   [Radiance 0.670 micron]
     4  -1.772  0.163E-01      234   256.00  1.000   [Radiance 0.670 micron]
     5  -2.029  0.164E-01      234   256.00  1.000   [Radiance 0.670 micron]
     6  -2.192  0.122E-01      297   256.00  1.000   [Radiance 0.670 micron]
     7  -2.193  0.495E-02      387   256.00  1.000   [Radiance 0.670 micron]
     8  -2.556  0.419E-02      396   256.00  1.000   [Radiance 0.670 micron]
Further cell splitting requires larger array allocations:  Ncells=    1012  Npnts=     578
     9  -2.728  0.267E-02      578   256.00  1.000   [Radiance 0.670 micron]
    10  -2.782  0.267E-02      578   256.00  1.000   [Radiance 0.670 micron]
    11  -2.979  0.268E-02      578   256.00  1.000   [Radiance 0.

/data/keeling/a/johntl3/miniconda3/envs/at3d_pure_k9/lib/python3.10/site-packages/at3d/solver.py:2337: UserWarning: Number of SHDOM grid cells with optical depth greater than 2: '8'. Max cell optical depth: '2.239902973175049'
  warnings.warn("Number of SHDOM grid cells with optical depth greater than 2: '{}'. "


  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -0.882  0.180E+00      135   256.00  1.000   [Radiance 0.670 micron]
     2  -1.451  0.189E+00      135   256.00  1.000   [Radiance 0.670 micron]
     3  -1.616  0.201E-01      216   256.00  1.000   [Radiance 0.670 micron]
     4  -1.786  0.180E-01      234   256.00  1.000   [Radiance 0.670 micron]
     5  -2.036  0.177E-01      234   256.00  1.000   [Radiance 0.670 micron]
     6  -2.193  0.123E-01      306   256.00  1.000   [Radiance 0.670 micron]
     7  -2.157  0.558E-02      387   256.00  1.000   [Radiance 0.670 micron]
     8  -2.548  0.454E-02      396   256.00  1.000   [Radiance 0.670 micron]
Further cell splitting requires larger array allocations:  Ncells=    1012  Npnts=     578
     9  -2.752  0.282E-02      578   256.00  1.000   [Radiance 0.670 micron]
    10  -2.807  0.283E-02      578   256.00  1.000   [Radiance 0.670 micron]
    11  -2.970  0.284E-02      578   256.00  1.000   [Radiance 0.

...

  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -0.940  0.591E+00      135   256.00  1.000   [Radiance 0.670 micron]
     2  -1.704  0.587E+00      135   256.00  1.000   [Radiance 0.670 micron]
     3  -1.497  0.300E-01      378   256.00  1.000   [Radiance 0.670 micron]
     4  -1.465  0.303E-01      432   256.00  1.000   [Radiance 0.670 micron]
     5  -1.729  0.319E-01      432   256.00  1.000   [Radiance 0.670 micron]
     6  -1.876  0.297E-01      441   256.00  1.000   [Radiance 0.670 micron]
     7  -2.039  0.260E-01      450   256.00  1.000   [Radiance 0.670 micron]
Further cell splitting requires larger array allocations:  Ncells=    1012  Npnts=     578
     8  -2.156  0.182E-01      578   256.00  1.000   [Radiance 0.670 micron]
     9  -2.211  0.185E-01      578   256.00  1.000   [Radiance 0.670 micron]
    10  -2.426  0.186E-01      578   256.00  1.000   [Radiance 0.670 micron]
    11  -2.542  0.189E-01      578   256.00  1.000   [Radiance 0.

/data/keeling/a/johntl3/miniconda3/envs/at3d_pure_k9/lib/python3.10/site-packages/at3d/solver.py:2337: UserWarning: Number of SHDOM grid cells with optical depth greater than 2: '16'. Max cell optical depth: '38.148677825927734'
  warnings.warn("Number of SHDOM grid cells with optical depth greater than 2: '{}'. "


  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -0.940  0.592E+00      135   256.00  1.000   [Radiance 0.670 micron]
     2  -1.704  0.589E+00      135   256.00  1.000   [Radiance 0.670 micron]
     3  -1.493  0.300E-01      378   256.00  1.000   [Radiance 0.670 micron]
     4  -1.463  0.303E-01      432   256.00  1.000   [Radiance 0.670 micron]
     5  -1.727  0.318E-01      432   256.00  1.000   [Radiance 0.670 micron]
     6  -1.876  0.331E-01      432   256.00  1.000   [Radiance 0.670 micron]
     7  -2.043  0.263E-01      450   256.00  1.000   [Radiance 0.670 micron]
Further cell splitting requires larger array allocations:  Ncells=    1012  Npnts=     578
     8  -2.155  0.192E-01      578   256.00  1.000   [Radiance 0.670 micron]
     9  -2.180  0.195E-01      578   256.00  1.000   [Radiance 0.670 micron]
    10  -2.399  0.196E-01      578   256.00  1.000   [Radiance 0.670 micron]
    11  -2.506  0.200E-01      578   256.00  1.000   [Radiance 0.

/data/keeling/a/johntl3/miniconda3/envs/at3d_pure_k9/lib/python3.10/site-packages/at3d/solver.py:2337: UserWarning: Number of SHDOM grid cells with optical depth greater than 2: '16'. Max cell optical depth: '41.193172454833984'
  warnings.warn("Number of SHDOM grid cells with optical depth greater than 2: '{}'. "


  ! Iter Log(Sol)  SplitCrit  Npoints  Nsh(avg)   [Radiance 0.670 micron]
     1  -0.941  0.594E+00      135   256.00  1.000   [Radiance 0.670 micron]
     2  -1.703  0.591E+00      135   256.00  1.000   [Radiance 0.670 micron]
     3  -1.489  0.300E-01      378   256.00  1.000   [Radiance 0.670 micron]
     4  -1.461  0.302E-01      432   256.00  1.000   [Radiance 0.670 micron]
     5  -1.725  0.315E-01      432   256.00  1.000   [Radiance 0.670 micron]
     6  -1.875  0.328E-01      432   256.00  1.000   [Radiance 0.670 micron]
     7  -2.045  0.267E-01      450   256.00  1.000   [Radiance 0.670 micron]
Further cell splitting requires larger array allocations:  Ncells=    1012  Npnts=     578
     8  -2.152  0.201E-01      578   256.00  1.000   [Radiance 0.670 micron]
     9  -2.144  0.206E-01      578   256.00  1.000   [Radiance 0.670 micron]
    10  -2.376  0.207E-01      578   256.00  1.000   [Radiance 0.670 micron]
    11  -2.482  0.211E-01      578   256.00  1.000   [Radiance 0.

In [29]:
solvers[0.67].numerical_params.num_stokes.data

array(1)

In [7]:
LUT = xr.Dataset(
    coords = dict(I=Is),
    data_vars = dict(
        tau=(['I'], taus),
        num_unique = (['I'], num_unique),
        diff = (['I'], diff)
    )
)
assert np.all(np.diff(LUT.tau.data) > 0), "LUT tau not monotonic"
assert np.all(np.diff(LUT.I.data) > 0), "LUT I not monotonic"
if(save_LUT):
    LUT.to_netcdf(LUT_outfile)